# Chuard — modelos, calibración y evaluación

Comparamos cinco familias con/sin seis características derivadas: diez candidatos.
Este notebook documenta el experimento y su reproducción; **no entrena, no cambia
el ganador ni vuelve a evaluar la prueba reservada**. Reutiliza `src/`.

El repositorio incluye el ganador, los reportes verificados y la reproducción que
consume este notebook. El CSV original se obtiene siguiendo el README. Para repetir
la búsqueda sin tocar el ganador ni prueba, usar
`python -m src.reproduce --output-dir <carpeta-nueva> --jobs 2`.
Las fuentes y copias ejecutadas de los notebooks se incluyen en la entrega académica.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Abra el notebook desde la raíz de Chuard o notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import EVALUATION_REPORT_DIR, MODEL_PATH, REPRODUCTION_DIR
from src.data import dataset_sha256
from src.reporting import generate_frozen_reports, load_frozen_resources
from src.reproduce import compare_reproduction

resources = load_frozen_resources()
meta = resources.manifest
frozen_hash = dataset_sha256(MODEL_PATH)
display(
    Markdown(
        f"**Ganador congelado:** {meta['model_name']}, versión "
        f"{meta['model_version']}; umbral {meta['decision_threshold']:.2f}."
    )
)
display(pd.Series(meta["protocol"], name="Protocolo registrado").to_frame())

## Particiones y prevención de fuga

4.225 filas de entrenamiento, 1.409 de validación y 1.409 de prueba; semilla 42.
Las 19 entradas excluyen ID y etiqueta. Imputación, escalado y One-Hot se ajustan
dentro del pipeline de cada fold. Las derivadas se calculan por fila dentro del pipeline.

Cada candidato usa RandomizedSearchCV con 20 combinaciones y cinco folds, optimizando
F1 positivo. Con los parámetros elegidos se generan probabilidades fuera de fold en
entrenamiento para estudiar calibración: sigmoid se adopta si Brier y log-loss mejoran
al menos 1%; SVM siempre necesita Platt para obtener probabilidades. El umbral se
elige en validación sobre la rejilla fija 0,05–0,95, paso 0,01.

Las estimaciones OOF para calibración reutilizan hiperparámetros seleccionados con CV
del entrenamiento; no equivalen a una evaluación externa anidada del proceso completo.
La comparación de modelos y umbrales en validación también introduce optimismo.

In [ ]:
provenance = json.loads((REPRODUCTION_DIR / "reproduction_provenance.json").read_text("utf-8"))
experiment = json.loads((REPRODUCTION_DIR / "experiment.json").read_text("utf-8"))
reproduced = pd.read_csv(REPRODUCTION_DIR / "metricas_modelos.csv")
search = pd.read_csv(REPRODUCTION_DIR / "search_cv_results.csv")
comparison = compare_reproduction(resources.comparison, reproduced)
assert provenance["all_candidates_match"] and comparison["all_candidates_match"]
assert provenance["immutable_inputs_verified"]
assert not provenance["test_rows_available_to_experiment"]
assert experiment["winner"] == meta["model_name"]
assert experiment["dataset_sha256"] == meta["dataset_sha256"]
assert experiment["split_manifest_sha256"] == meta["split_manifest_sha256"]
assert len(search) == 200 and search["candidate"].nunique() == 10
assert search.filter(regex="^cv_").notna().all().all()
display(pd.DataFrame(comparison["candidates"])[["candidate", "matches"]])
print("Reproducción independiente completada:", provenance["completed_at_utc"])
print("200 combinaciones evaluadas, 5 folds por combinación; no se usaron filas de prueba.")
print("Estos archivos nuevos no son los originales desaparecidos del experimento anterior.")

## Comparación en validación

Accuracy es la fracción de aciertos; precisión, la fracción de positivos predichos que
realmente abandonaron; recall, la fracción de abandonos detectados. F1 combina
precisión y recall. ROC-AUC mide ordenación y AP resume precisión-recall sin ser
el área trapezoidal bajo esa curva. Brier y log-loss evalúan las probabilidades.
Son métricas históricas de este dataset, no promesas comerciales.

In [ ]:
columns = [
    "candidate",
    "calibration",
    "decision_threshold",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision",
    "brier",
    "log_loss",
]
display(resources.comparison[columns].sort_values("f1", ascending=False).round(5))
display(reproduced.loc[~reproduced.elegible].round(5))
variants = resources.comparison.assign(
    familia=lambda frame: frame.candidate.str.split("__").str[0],
    variante=lambda frame: frame.candidate.str.split("__").str[1],
).pivot(index="familia", columns="variante", values="f1")
variants["delta_F1_con_menos_sin"] = variants.con_derivadas - variants.sin_derivadas
display(variants.round(5))
display(
    Markdown(
        "El efecto de las derivadas depende de la familia; no hay una mejora "
        "uniforme. El ganador registrado conserva las entradas originales."
    )
)

In [ ]:
candidates = [candidate for candidate in experiment["candidates"] if candidate["eligible"]]
cv = pd.DataFrame(
    [
        {
            "candidato": c["key"],
            "F1_CV_media": c["cv"]["f1"]["mean"],
            "F1_CV_desviacion": c["cv"]["f1"]["std"],
            "calibracion": c["calibration"],
            "Brier_OOF_sin": c["oof_calibration"].get("sin_calibrar", {}).get("brier"),
            "Brier_OOF_sigmoid": c["oof_calibration"]["sigmoid"]["brier"],
            "log_loss_OOF_sin": c["oof_calibration"].get("sin_calibrar", {}).get("log_loss"),
            "log_loss_OOF_sigmoid": c["oof_calibration"]["sigmoid"]["log_loss"],
            "combinaciones_fallidas": c["failed_parameter_sets"],
        }
        for c in candidates
    ]
)
display(cv.round(6))
winner = next(c for c in candidates if c["key"] == meta["model_name"])
assert winner["params"] == meta["hyperparameters"]
for metric in ("f1", "roc_auc", "recall"):
    np.testing.assert_allclose(
        winner["cv"][metric]["folds"],
        meta["metrics"]["cv_train"][metric]["folds"],
        atol=1e-12,
        rtol=0,
    )
display(
    pd.DataFrame(
        {metric: winner["cv"][metric]["folds"] for metric in ("f1", "roc_auc", "recall")},
        index=range(1, 6),
    )
)
display(pd.Series(winner["params"], name="Hiperparámetros congelados").to_frame())

## Selección y curvas

Random Forest sin derivadas tiene el mayor F1 de validación. La regla registrada
considera empate operativo las diferencias de hasta 0,005 y desempata por ROC-AUC,
recall, estabilidad CV y parsimonia. XGBoost permanece dentro de la tolerancia de
F1 y ROC-AUC y gana por recall. Esa tolerancia no demuestra equivalencia estadística.
No se cambió la regla después de observar prueba.

In [ ]:
for step in meta["selection_trace"]:
    print(step)
verification = generate_frozen_reports(resources)
assert verification["validation_reproduced"] and not verification["test_reevaluated"]
for filename, directory in (
    ("comparacion_validacion.png", EVALUATION_REPORT_DIR),
    ("roc_validacion.png", REPRODUCTION_DIR / "figures"),
    ("calibracion_entrenamiento_oof.png", REPRODUCTION_DIR / "figures"),
    ("matriz_validacion_ganador.png", REPRODUCTION_DIR / "figures"),
):
    display(Image(filename=str(directory / filename)))
display(
    Markdown(
        "Las curvas comparativas y OOF proceden de la reproducción. "
        "La tabla y la selección se contrastaron con los reportes originales."
    )
)

## Prueba final ya realizada

Las cifras siguientes se leen del CSV original, cuya huella coincide con la del
manifiesto. **No se llama a `run_final`, no se predice sobre prueba y no se reajusta
el modelo.** Se dibuja su matriz a partir de los conteos guardados. Las probabilidades
originales de prueba no están disponibles: no se reconstruye una curva ROC desde AUC.

In [ ]:
final = resources.final_report.iloc[0]
display(resources.final_report.round(6))
display(Image(filename=str(EVALUATION_REPORT_DIR / "matriz_prueba_final_guardada.png")))
display(
    Markdown(
        f"De {int(final['positives'])} abandonos históricos se detectaron "
        f"{int(final['true_positives'])}; quedaron {int(final['false_negatives'])} sin detectar. "
        f"Hubo {int(final['false_positives'])} falsos positivos: contactos potencialmente "
        "innecesarios si se utilizara la decisión binaria para una campaña."
    )
)
assert dataset_sha256(MODEL_PATH) == frozen_hash
for limitation in verification["limitations"]:
    print("-", limitation)

## Alcance de la entrega

Las bandas Bajo/Medio/Alto (0,35/0,65) siguen siendo convenciones operativas,
distintas del umbral binario 0,25. No se inventaron costes ni una optimización
comercial de bandas. El dataset no valida horizontes futuros ni eficacia de retención.
Los archivos originales de búsqueda faltan; la reproducción coincide numéricamente
con la comparación guardada, pero no acredita por sí sola todo el historial original.

El frente 3 conserva interfaz web, recomendaciones, aceptación, informe final y despliegue.